# TFScope (v24 ensemble): protein DBD → PWM → consensus → logo

Given a **DNA-binding-domain (DBD) protein sequence**, this notebook uses the **5-seed v24 ensemble** (`seed42 + seeds 1/7/13/23`) to predict the binding-motif PWM, print the consensus, and draw the sequence logo.

**Ensembling:** each member commits to its own motif core (span gate); members are register-aligned (offset + reverse-complement) to seed42's core, then averaged — the same protocol used to score the ensemble (full-291 content-r 0.664 vs single-seed 0.629).

**Important:** v24 is trained on **DBD crops** (~40–170 aa), not full-length proteins. Feed the DBD region only.

In [ ]:
import os, sys, json
# Pin to ONE good GPU *before* importing torch. This node has a broken GPU 9 that
# crashes torch's capability check when all GPUs are visible (the CUDA "device=9"
# error). Set to a free index; use '' for CPU. If you already hit the error,
# RESTART THE KERNEL and run from this cell.
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ.setdefault('TORCH_HOME', '/data1/leihuang/.cache/torch')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
sys.path.insert(0, '../src'); sys.path.insert(0, 'src'); sys.path.insert(0, '..'); sys.path.insert(0, '.')
import numpy as np, torch, torch.nn.functional as F
import matplotlib.pyplot as plt, logomaker, pandas as pd
from tfscope.config import TFScopeConfig
from tfscope.models.tfscope import TFScopeModel
from tfscope.data.dataset import AA_TO_TOKEN
from tfscope.models.alignment import align_pwm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BASES = np.array(list('ACGT'))
# 5 ensemble members (seed42 is the reference frame)
CKPTS = ['/data1/leihuang/project/TFScope/checkpoints/v24_contact/contact_v24_seed42'] + \
        [f'../checkpoints/iclr_phase1/v24_ens/seed{s}' for s in (1, 7, 13, 23)]
print('device:', DEVICE, '| members:', len(CKPTS))

In [ ]:
# ---- load all 5 ensemble members once ----
def load_member(ckpt_dir):
    cfg = TFScopeConfig()
    for k, v in json.load(open(f'{ckpt_dir}/config.json')).items():
        if hasattr(cfg, k):
            try: setattr(cfg, k, type(getattr(cfg, k))(v))
            except Exception: setattr(cfg, k, v)
    cfg.use_retrieval = False
    m = TFScopeModel(cfg).to(DEVICE).eval()
    sd = torch.load(f'{ckpt_dir}/ckpt_best.pt', map_location=DEVICE, weights_only=False)
    m.load_state_dict(sd.get('model', sd), strict=False)
    return m

MEMBERS = [load_member(c) for c in CKPTS]
print(f'loaded {len(MEMBERS)}-model v24 ensemble')

In [ ]:
@torch.no_grad()
def member_core(model, seq, family_id):
    """One member's committed motif core (4, L) via its span gate."""
    tok = torch.tensor([[AA_TO_TOKEN.get(a, 4) for a in seq]], dtype=torch.long, device=DEVICE)
    dbd = torch.ones(1, len(seq), dtype=torch.bool, device=DEVICE)
    fid = torch.tensor([int(family_id)], device=DEVICE)
    gl, pl, aux = model(tok, dbd, fid, retrieved_pwms=None, retrieved_masks=None,
                        retrieved_sims=None, recog_prior=None)
    P = F.softmax(pl, 1)[0].cpu().numpy()                       # (4, 42)
    if aux.get('span_start') is not None and aux.get('span_length') is not None:
        s = int(round(float(np.asarray(aux['span_start'].cpu()).reshape(-1)[0])))
        l = int(round(float(np.asarray(aux['span_length'].cpu()).reshape(-1)[0])))
        s = max(0, min(s, P.shape[1] - 1)); l = max(1, min(l, P.shape[1] - s))
        return P[:, s:s + l]
    gate = gl.sigmoid()[0].cpu().numpy(); L = max(4, int((gate > 0.5).sum()))
    return P[:, :L]

@torch.no_grad()
def predict_pwm(seq, family_id=0):
    """5-seed ensemble: register each member's core to seed42's, average -> (core 4xL, consensus)."""
    seq = seq.strip().upper()
    cores = [member_core(m, seq, family_id) for m in MEMBERS]
    ref = cores[0]                                              # seed42 frame
    stack = [ref]
    for c in cores[1:]:
        aligned, _, _, _ = align_pwm(c, ref, max_shift=10, consider_revcomp=True, min_overlap=3)
        stack.append(aligned)                                  # (4, len(ref))
    cons = np.mean(np.stack(stack, 0), 0)
    cons = cons / np.clip(cons.sum(0, keepdims=True), 1e-8, None)
    return cons, ''.join(BASES[cons.argmax(0)])

def plot_logo(core, title='TFScope v24-ensemble predicted motif'):
    P = np.clip(core.T, 1e-9, 1.0)                              # (L, 4)
    ic = (P * np.log2(P / 0.25)).sum(1, keepdims=True)          # bits/position
    df = pd.DataFrame(P * ic, columns=list('ACGT'))
    fig, ax = plt.subplots(figsize=(max(3, 0.6 * len(df)), 2.2))
    logomaker.Logo(df, ax=ax, color_scheme='classic')
    ax.set_ylabel('bits'); ax.set_ylim(0, 2); ax.set_title(title)
    ax.set_xticks(range(len(df))); ax.set_xticklabels(range(1, len(df) + 1))
    plt.tight_layout(); plt.show()
print('helpers ready: predict_pwm(seq, family_id) [5-model ensemble], plot_logo(core)')

## Example: MyoD1 bHLH DBD
MyoD1's basic-helix-loop-helix domain binds the E-box `CANNTG` (muscle E-box `CACCTG`/`CAGCTG`). `family_id=3` = bHLH.

In [ ]:
MYOD1_DBD = 'RKAATMRERRRLSKVNEAFETLKRCTSSNPNQRLPKVEILRNAIRYIEGLQA'
core, cons = predict_pwm(MYOD1_DBD, family_id=3)
print('consensus:', cons, '  (motif length', core.shape[1], 'bp)')
plot_logo(core, title=f'MyoD1 bHLH — v24 ensemble ({cons})')

## Your own protein
Paste a **DBD-cropped** sequence. The family head is near-inert in v24, so `family_id` barely changes the prediction; use a sensible family or 0.

In [ ]:
MY_SEQ = 'RKAATMRERRRLSKVNEAFETLKRCTSSNPNQRLPKVEILRNAIRYIEGLQA'  # <-- replace with your DBD
MY_FAMILY = 3
core, cons = predict_pwm(MY_SEQ, family_id=MY_FAMILY)
print('consensus:', cons, '  length', core.shape[1])
print('PWM (rows A,C,G,T):'); print(np.round(core, 3))
plot_logo(core, title=f'v24 ensemble — predicted motif ({cons})')